# LangChain과 Chroma를 활용한 RAG 구성

1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰수 초과로 답변을 생성하지 못할 수도 있고
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래걸림
    - split 된 데이터 chunk를 Large Language Model(LLM)에게 전달하면 토큰 절약 가능
    - 비용 감소와 답변 생성시간 감소의 효과
3. 임베딩 --> 벡터 DB에 저장
4. 질문이 있을 때, 벡터 DB에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

## 1. 문서 쪼개서 document_list 생성
- 마크다운 파일 1개를 1개의 document 객체로 만들어 리스트 반환 (문서 쪼개는 것 대신)

In [3]:
'''
import os
import json
import re
from langchain.schema import Document # langchain_core.documents 로 변경 가능

def load_documents_from_metadata_json(
    md_dir="md_pdf",
    metadata_filepath="all_metadata.json"
):
    """
    all_metadata.json 파일을 기반으로 문서를 생성합니다.
    JSON 안의 각 메타데이터에 해당하는 md 파일의 내용을 읽어와
    images 정보를 포함한 전체 메타데이터와 함께 Document 객체를 만듭니다.
    """
    # 1. 먼저 all_metadata.json 파일을 읽어옵니다.
    with open(metadata_filepath, "r", encoding="utf-8") as f:
        all_metadata = json.load(f)

    document_list = []
    # 2. 파일 목록이 아닌, JSON 안의 메타데이터 목록을 순회합니다.
    for metadata in all_metadata:
        md_file_name = metadata.get("source")
        if not md_file_name:
            continue

        file_path = os.path.join(md_dir, md_file_name)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()

            # 3. 파일 내용과 함께, JSON에서 가져온 '완전한' 메타데이터를 사용합니다.
            document_list.append(
                Document(
                    page_content=content,
                    metadata=metadata  # images 정보가 포함된 전체 메타데이터
                )
            )
        except FileNotFoundError:
            print(f"⚠️  경고: {file_path} 파일을 찾을 수 없습니다. 건너뜁니다.")
            continue

    print(f"✅ Document 개수: {len(document_list)}")
    return document_list


# ===== 실행 예시 =====
if __name__ == "__main__":
    # 새로운 함수를 호출합니다.
    document_list = load_documents_from_metadata_json(
        md_dir="md_pdf",
        metadata_filepath="all_metadata.json"
    )
'''

✅ Document 개수: 49


- hwp 불러오기

In [2]:
from langchain_core.documents import Document
import json

def load_docs_from_json(input_path="split_hwp.json"):
    """
    JSON 파일을 langchain_core.documents.Document 리스트로 복원.
    """
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    document_list = [
        Document(page_content=item["page_content"], metadata=item["metadata"])
        for item in data
    ]
    print(f"✅ {len(document_list)}개의 문서를 불러왔습니다.")
    return document_list

In [4]:
document_list = load_docs_from_json("split_hwp.json")

✅ 34개의 문서를 불러왔습니다.


## 2. 임베딩

In [1]:
# 3. 임베딩 해주기 --> 백터DB에 저장 (Chroma 쓸꺼고 / 인메모리라 간단)
from dotenv import load_dotenv # 임베딩에 openai key 인자가 있어서 환경변수를 로드해줘야 함
from langchain_openai import OpenAIEmbeddings 

# 환경변수 불러옴
load_dotenv()

# OpenAI에서 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large') # 임베딩 모델을 large로 바꿔주기

## 3. 벡터DB 생성 (Pinecone) 

In [ ]:
'''
# Pinecone 설치
%pip install --upgrade --quiet \
    langchain-pinecone \
    langchain-openai \
    langchain
'''

In [2]:
import os

from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name = 'finance-new-index' # finance-new-index : 요약매뉴얼+정산지침 / finance-index: 매뉴얼ppt+정산지침
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# 데이터를 추가(insert)할 때는 `from_documents()`
# 데이터를 추가한 이후에는 `from_existing_index()`를 사용 (데이터 추가없이 검색만)
#database = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name)
database = PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embedding)

/Users/dayeayim/.pyenv/versions/inflearn-streamlit/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import os
from pinecone import Pinecone

index_name = "finance-new-index"
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# 1️⃣ 인덱스 상태 확인
index_info = pc.describe_index(index_name)
print(f"📊 인덱스 상태: {index_info.status}")

# 2️⃣ 인덱스 통계(벡터 개수) 확인
index_stats = pc.Index(index_name).describe_index_stats()
print(f"📦 '{index_name}' 인덱스 요약:")
print(index_stats)

# 3️⃣ 깔끔하게 개수만 출력하고 싶다면
vector_count = index_stats["total_vector_count"]
print(f"✅ 현재 '{index_name}'에는 {vector_count:,}개의 벡터가 저장되어 있습니다.") # 83개

📊 인덱스 상태: {'ready': True, 'state': 'Ready'}
📦 'finance-new-index' 인덱스 요약:
{'dimension': 3072,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 83}},
 'total_vector_count': 83,
 'vector_type': 'dense'}
✅ 현재 'finance-new-index'에는 83개의 벡터가 저장되어 있습니다.


## 4. 문서 검색 및 답변
- 벡터DB에서 적합한 문서 찾고 LLM에 문서와 질의를 주면서 답변을 요청(RetrievalQA)
- RetrievalQA: 데이터를 검색(Retrieval)한 다음에 질문(Question)하고 답변(Answer) 할 것이다

In [23]:
# 문서를 가지고 왔으니까 질의를 해봐야 한다 
from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(model = 'gpt-4o')
#llm = ChatOpenAI(model = 'gpt-5')

In [24]:
%pip install --upgrade --force-reinstall \
  "langchain==0.3.27" \
  "langchain-core==0.3.79" \
  "langchain-openai==0.3.35" \
  "langchain-pinecone==0.2.11" \
  "langchain-text-splitters==0.3.11" \
  "langchain-upstage==0.7.4"

  Using cached langchain-0.3.27-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_core-0.3.79-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_openai-0.3.35-py3-none-any.whl.metadata (2.4 kB)
  Using cached langchain_pinecone-0.2.11-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_upstage-0.7.4-py3-none-any.whl.metadata (3.3 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached openai-2.5.0-py3-none-any.whl.metadata (29 kB)
  Using cached pinecone-7.3.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached pypdf-4.3.1-py3-none-any.whl.metadata (7.4 kB)
  Using cached tokenizers-0.20.3-cp312-cp312-macosx_10_12_x86_64.whl.metadata (6.7 kB)
  Using cached jsonpo

In [18]:
# Retrieval된 데이터는 LangChain에서 제공하는 프롬프트("rlm/rag-prompt") 사용해서 답변해보기
from langchain import hub 
from langchain.prompts import ChatPromptTemplate

# 기본 RAG 프롬프트 가져오기
base_prompt = hub.pull("rlm/rag-prompt")

# 시스템 역할 지시 추가해서 새 템플릿 만들기
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 재무팀에서 근무하는 해외무역관 정산 전문가입니다. "
               "[context]를 참고해서 사용자의 질문에 답변해주세요. "
               "[context]에 없는 정보는 지어내지 말고, 자료에 없음이라고 답하세요."),
    *base_prompt.messages  # 기존 RAG 메시지 구조 유지
])

In [19]:
# 답변생성 (QA 체인 만들기)
#-- RetrievalQA를 통해 LLM에 전달
#-- RetrievalQA는 create_retrieval_chain으로 대체됨
#-- 실제 ChatBot 구현 시 create_retrieval_chain으로 변경하는 과정을 볼 수 있음

from langchain.chains import RetrievalQA 

retriever = database.as_retriever( # 벡터DB
    search_kwargs={"k": 2}) # 유사도로 몇 개 문서 찾을 것인지
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = retriever, # 위에서 만든 retriever (DB에서 문서 검색)
    chain_type_kwargs = {"prompt": prompt}, # 위에서 만든 prompt (정산 전문가)
    return_source_documents=True # 문서 소스 포함
)

In [20]:
query = '예산이 아직 지급 안됐는데 예산을 미리 써야 하는 경우는 어떻게 해야해?'

In [21]:
ai_message = qa_chain.invoke({"query": query})

# 답변 본문
answer = ai_message["result"]

In [22]:
answer

'본사 예산지원이 결정됐으나 아직 미지급인 경우, 본사 승인 획득 후(자금부족 사전보고·차 분기 송금 확인) 관장 책임 하에 소유지불(선집행)로 처리합니다. 해당 분기에는 본사 승인공문 문서번호를 기재한 영수증을 첨부·보고하고, 현지화 집행분은 ERP에 현지화 금액과 송금통화 확정값을 동시에 입력합니다. 차 분기 정산 시 소유지불부터 우선 해제하여 지불결의서 및 전도자금 출납명세서에 반영해 정산을 완료합니다.'

In [10]:
ai_message["source_documents"]

[Document(id='96d68704-f8f8-4df1-af41-b6c39ad29922', metadata={'origin_pdf': '해외조직망정산지침', 'page_num': '6조. 정산업무 개요 및 정산서 작성'}, page_content='6조.  정산업무 개요 및 정산서 작성3.  소유지불 처리 가. 자금은 원칙적으로 본사에서 지원한 예산한도 내에서 집행하여야 한다. 단, 본사의 승인을 득한 경우에 한해 본사의 예산지원이 결정되었으나 소요예산이 지원되지 아니한 경우 관장 책임 하에 예산 확보 전에 소요비용을 소유지불(선집행)할 수 있다. 나. 어떠한 경우에도 소유지불을 처리할 때에는 소유지불을 처리하기 전에 본사(관련팀)에 자금부족 사실을 보고하여 차 분기 자금송금을 확인받은 후 처리해야 하며 차 분기 자금집행 시 우선 반영하여 소유지불을 해소하여야 한다. 다. 소유지불이 발생한 당해분기에는 본사의 승인공문 문서번호를 기재한 영수증을 첨부하여 보고하여야 한다. 라. 현지화로 집행한 소유지불 금액을 송금통화로 확정하기 위한 환율은 집행된 자금의 성격에 따라 입금(환전)환율 또는 평균환율을 적용하여 ERP재무회계시스템 회계장부에 현지화 금액뿐만 아니라 송금통화 확정값을 동시에 입력하여야 한다. 마. 차 분기 정산 시 소유지불로 설정한 금액을 우선적으로 해제하여야 하며, 소유지불을 해제하면 해당 사업 및 비목의 지불결의서에 자동으로 반영되어 전도자금 출납명세서에 본 분기 지불액에 합산되어 정산이 완료된다.'),
 Document(id='63f39701-8816-49bf-98ca-d25fd683ad77', metadata={'origin_pdf': '해외조직망정산지침', 'page_num': '9조. 정산 시 유의사항 (공통사항)'}, page_content='9조.  정산 시 유의사항 (공통사항)  6.  기 타 가. 본사가 증빙서만으로 이해하기 곤란하다고 판단되는 자금을 집행한 경우 증빙서 여백에 자금집행내역을 상세히 기재하여야 한다. 나. 정산서 

In [14]:
# 강의에서는 위처럼 진행하지만 업데이트된 LangChain 문법은 `.invoke()` 활용을 권장 
# 객체에서는 속성에 점(.)을 사용하여 접근
ai_message = qa_chain.invoke({"query": query})

# 답변 본문
answer = ai_message["result"]

# 페이지(조 단위 제목) 추출
pages = [doc.metadata["page_num"] for doc in ai_message["source_documents"]]
pages = sorted(set(pages))  # 중복 제거 + 정렬 (문자열 정렬)
origin = ai_message["source_documents"][0].metadata["origin_pdf"]

# 최종 출력
print(answer)
print(f"📖 출처: {origin} / 상세: {', '.join(map(str, pages))}")

예산이 지급되지 않았지만 미리 사용해야 할 경우, 본사의 승인을 받아 소유지불(선집행)을 할 수 있습니다. 소유지불을 처리하기 전, 본사에 자금 부족 사실을 보고하고 차 분기 자금 송금을 확인받아야 합니다. 이후, 정산 시 본사의 승인 공문 문서 번호가 기재된 영수증을 첨부하여 보고해야 합니다.
📖 출처: 해외조직망정산지침 상세: 6조. 정산업무 개요 및 정산서 작성, 9조. 정산 시 유의사항 (공통사항)


In [12]:
# 어떤 문서가 검색되었는지 확인
retriever.invoke(query)

[Document(id='96d68704-f8f8-4df1-af41-b6c39ad29922', metadata={'origin_pdf': '해외조직망정산지침', 'page_num': '6조. 정산업무 개요 및 정산서 작성'}, page_content='6조.  정산업무 개요 및 정산서 작성3.  소유지불 처리 가. 자금은 원칙적으로 본사에서 지원한 예산한도 내에서 집행하여야 한다. 단, 본사의 승인을 득한 경우에 한해 본사의 예산지원이 결정되었으나 소요예산이 지원되지 아니한 경우 관장 책임 하에 예산 확보 전에 소요비용을 소유지불(선집행)할 수 있다. 나. 어떠한 경우에도 소유지불을 처리할 때에는 소유지불을 처리하기 전에 본사(관련팀)에 자금부족 사실을 보고하여 차 분기 자금송금을 확인받은 후 처리해야 하며 차 분기 자금집행 시 우선 반영하여 소유지불을 해소하여야 한다. 다. 소유지불이 발생한 당해분기에는 본사의 승인공문 문서번호를 기재한 영수증을 첨부하여 보고하여야 한다. 라. 현지화로 집행한 소유지불 금액을 송금통화로 확정하기 위한 환율은 집행된 자금의 성격에 따라 입금(환전)환율 또는 평균환율을 적용하여 ERP재무회계시스템 회계장부에 현지화 금액뿐만 아니라 송금통화 확정값을 동시에 입력하여야 한다. 마. 차 분기 정산 시 소유지불로 설정한 금액을 우선적으로 해제하여야 하며, 소유지불을 해제하면 해당 사업 및 비목의 지불결의서에 자동으로 반영되어 전도자금 출납명세서에 본 분기 지불액에 합산되어 정산이 완료된다.'),
 Document(id='63f39701-8816-49bf-98ca-d25fd683ad77', metadata={'origin_pdf': '해외조직망정산지침', 'page_num': '9조. 정산 시 유의사항 (공통사항)'}, page_content='9조.  정산 시 유의사항 (공통사항)  6.  기 타 가. 본사가 증빙서만으로 이해하기 곤란하다고 판단되는 자금을 집행한 경우 증빙서 여백에 자금집행내역을 상세히 기재하여야 한다. 나. 정산서 

## 5. Retrieval을 위한 keyword 사전 활용
- 직장인이라는 질의가 들어오면, 직장인을 거주자로 자동으로 바꾸도록 설정
- Knowledge Base에서 사용되는 keyword를 활용하여 사용자 질문 수정
- LangChain Expression Language (LCEL)을 활용한 Chain 연계

In [51]:
query = '4직급의 기준을 자세하게 알려주세요. 보기좋게 연번으로 알려주세요.' # 표를 마크다운으로 바꾸면 더 잘 읽는지 확인

In [52]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary = ["기준을 나타내는 표현 -> 경력 기준"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요.
    사전: {dictionary}
    
    질문: {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()
tax_chain = {"query": dictionary_chain} | qa_chain

In [53]:
new_question = dictionary_chain.invoke({"question": query})

In [54]:
# 바뀐 질의
new_question

'4직급의 경력 기준을 자세하게 알려주세요. 보기 좋게 연번으로 알려주세요.'

In [26]:
ai_response = tax_chain.invoke({"question": query})

In [28]:
print(ai_response['result'])

4직급의 경력 기준은 다음과 같습니다:

1. 행정 분야에서 근무한 6급 공무원으로 5년 이상 경력 소지자
2. 정부투자기관, 경제단체 및 유관기관에서 동일직급 2년 이상 경력 소지자
3. 소령 2년 이상 경력 소지자
4. 민간기업 과장급으로 유관부문 2년 이상 경력 소지자
5. 대학 및 전문학교 전임강사 2년 이상 경력 소지자
6. 전문조사기관의 연구원으로 3년 이상 경력 소지자
7. 해당 분야에 실무경력, 연구 또는 연수 경력자로 당해 직급에 자격이 있다고 인사위원회에서 인정되는 자
8. 기타 전항과 동등한 자격이 있다고 인사위원회에서 인정되는 자
